In [1]:
"""
Exploratory joint real-data fit for DVCH and LCDM using:
- Pantheon+SH0ES supernova distances
- DESI DR1 BAO compressed measurements
- Cosmic chronometer H(z) data

This script is intended as a reproducibility prototype rather than a final MCMC
analysis. In the present implementation, Omega_r0 and beta are fixed to
fiducial values, and optimization is performed with Nelder-Mead.
"""

from __future__ import annotations

import io
from dataclasses import dataclass

import numpy as np
import pandas as pd
import requests
from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.optimize import minimize

C_LIGHT = 299792.458

PANTHEON_DATA_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon+_Data/4_DISTANCES_AND_COVAR/Pantheon+SH0ES.dat"
)
PANTHEON_COV_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon+_Data/4_DISTANCES_AND_COVAR/Pantheon+SH0ES_STAT+SYS.cov"
)

BAO_BLOCKS = [
    {"z": 0.295, "kind": "DV", "mean": 7.93, "sigma": 0.15},
    {"z": 0.510, "kind": "DM_DH", "mean": [13.62, 20.98], "sigma": [0.25, 0.61], "rho": -0.445},
    {"z": 0.706, "kind": "DM_DH", "mean": [16.85, 20.08], "sigma": [0.32, 0.60], "rho": -0.420},
    {"z": 0.930, "kind": "DM_DH", "mean": [21.71, 17.88], "sigma": [0.28, 0.35], "rho": -0.389},
    {"z": 1.317, "kind": "DM_DH", "mean": [27.79, 13.82], "sigma": [0.69, 0.42], "rho": -0.444},
    {"z": 1.491, "kind": "DV", "mean": 26.07, "sigma": 0.67},
    {"z": 2.330, "kind": "DM_DH", "mean": [39.71, 8.52], "sigma": [0.94, 0.17], "rho": -0.477},
]

HZ_CC = np.array([
    [0.07, 69.0, 19.6], [0.09, 69.0, 12.0], [0.12, 68.6, 26.2],
    [0.17, 83.0, 8.0], [0.1791, 78.0, 6.2], [0.1993, 78.0, 6.9],
    [0.20, 72.9, 29.6], [0.27, 77.0, 14.0], [0.28, 88.8, 36.6],
    [0.3519, 85.5, 15.7], [0.3802, 86.2, 14.6], [0.40, 95.0, 17.0],
    [0.4004, 79.9, 11.4], [0.4247, 90.4, 12.8], [0.4497, 96.3, 14.4],
    [0.47, 89.0, 49.6], [0.4783, 83.8, 10.2], [0.48, 97.0, 62.0],
    [0.5929, 107.0, 15.5], [0.6797, 95.0, 10.5], [0.75, 98.8, 33.6],
    [0.7812, 96.5, 12.5], [0.8754, 124.5, 17.4], [0.88, 90.0, 40.0],
    [0.90, 117.0, 23.0], [1.037, 133.5, 17.6], [1.30, 168.0, 17.0],
    [1.363, 160.0, 33.8], [1.43, 177.0, 18.0], [1.53, 140.0, 14.0],
    [1.75, 202.0, 40.0], [1.965, 186.5, 50.6],
], dtype=float)


@dataclass(frozen=True)
class FitConfig:
    omega_r0: float = 9e-5
    beta: float = 1e-4
    dvch_init: tuple[float, float, float, float, float] = (73.0, 0.30, 0.20, 147.0, 0.0)
    lcdm_init: tuple[float, float, float, float] = (73.0, 0.30, 147.0, 0.0)
    maxiter: int = 12000
    xatol: float = 1e-6
    fatol: float = 1e-6


def load_pantheon_plus():
    data_response = requests.get(PANTHEON_DATA_URL, timeout=60)
    cov_response = requests.get(PANTHEON_COV_URL, timeout=60)
    data_response.raise_for_status()
    cov_response.raise_for_status()

    dataframe = pd.read_csv(io.StringIO(data_response.text), sep=r"\s+", comment="#")

    cov_lines = cov_response.text.strip().splitlines()
    n_cov = int(cov_lines[0].strip())
    covariance = np.array([float(x) for x in cov_lines[1:]], dtype=float).reshape(n_cov, n_cov)

    mask = (dataframe["IS_CALIBRATOR"] == 0).to_numpy()
    dataframe = dataframe.loc[mask].reset_index(drop=True)
    covariance = covariance[np.ix_(mask, mask)]

    z = dataframe["zHD"].to_numpy(dtype=float)
    mu = dataframe["MU_SH0ES"].to_numpy(dtype=float)

    good = np.isfinite(z) & np.isfinite(mu) & (z > 0.0)
    z = z[good]
    mu = mu[good]
    covariance = covariance[np.ix_(good, good)]

    return z, mu, covariance


def rhs_dvch(z, y, n, beta):
    omega_m, omega_l, omega_r = y

    omega_m = max(omega_m, 1e-15)
    omega_l = max(omega_l, 0.0)
    omega_r = max(omega_r, 0.0)

    e2 = max(omega_m + omega_l + omega_r, 1e-20)
    e = np.sqrt(e2)

    qtilde = -e * omega_l / (1.0 + n * omega_l / omega_m) * (
        n - (beta / (1.0 + beta * e2)) * ((4.0 * omega_r + 3.0 * omega_m) / 3.0)
    )

    domega_m_dz = 3.0 * (omega_m + qtilde / e) / (1.0 + z)
    domega_l_dz = -3.0 * qtilde / (e * (1.0 + z))
    domega_r_dz = 4.0 * omega_r / (1.0 + z)

    return [domega_m_dz, domega_l_dz, domega_r_dz]


def solve_dvch_grid(zmax, H0, omega_m0, omega_r0, n, beta, npts=3500):
    omega_l0 = 1.0 - omega_m0 - omega_r0
    if omega_l0 <= 0.0:
        raise ValueError("Omega_Lambda0 must be positive.")

    z_grid = np.linspace(0.0, zmax, npts)
    solution = solve_ivp(
        lambda z, y: rhs_dvch(z, y, n=n, beta=beta),
        (0.0, zmax),
        [omega_m0, omega_l0, omega_r0],
        t_eval=z_grid,
        rtol=1e-9,
        atol=1e-12,
        method="RK45",
    )
    if not solution.success:
        raise RuntimeError(solution.message)

    omega_m, omega_l, omega_r = solution.y
    e_grid = np.sqrt(np.maximum(omega_m + omega_l + omega_r, 1e-20))
    return z_grid, e_grid


def solve_lcdm_grid(zmax, omega_m0, omega_r0, npts=3500):
    z_grid = np.linspace(0.0, zmax, npts)
    omega_l0 = 1.0 - omega_m0 - omega_r0
    e_grid = np.sqrt(
        omega_m0 * (1.0 + z_grid) ** 3
        + omega_r0 * (1.0 + z_grid) ** 4
        + omega_l0
    )
    return z_grid, e_grid


def distance_grids(z_grid, e_grid, H0):
    chi_grid = (C_LIGHT / H0) * cumulative_trapezoid(1.0 / e_grid, z_grid, initial=0.0)
    dh_grid = C_LIGHT / (H0 * e_grid)
    return chi_grid, dh_grid


def distance_modulus_from_grid(z_data, z_grid, chi_grid, M_shift):
    chi_at = np.interp(z_data, z_grid, chi_grid)
    d_l = (1.0 + z_data) * chi_at
    return 5.0 * np.log10(d_l) + 25.0 + M_shift


def pantheon_chi2(z_grid, e_grid, H0, M_shift, z_sn, mu_sn, covinv_sn):
    chi_grid, _ = distance_grids(z_grid, e_grid, H0)
    mu_model = distance_modulus_from_grid(z_sn, z_grid, chi_grid, M_shift)
    delta = mu_sn - mu_model
    return float(delta @ covinv_sn @ delta)


def bao_chi2(z_grid, e_grid, H0, rd):
    chi_grid, dh_grid = distance_grids(z_grid, e_grid, H0)
    total = 0.0

    for block in BAO_BLOCKS:
        z = block["z"]
        dm = np.interp(z, z_grid, chi_grid)
        dh = np.interp(z, z_grid, dh_grid)

        if block["kind"] == "DV":
            dv = (z * dm * dm * dh) ** (1.0 / 3.0)
            model = dv / rd
            total += ((model - block["mean"]) / block["sigma"]) ** 2
        else:
            model = np.array([dm / rd, dh / rd])
            mean = np.array(block["mean"])
            sigma1, sigma2 = block["sigma"]
            rho = block["rho"]
            covariance = np.array([
                [sigma1 * sigma1, rho * sigma1 * sigma2],
                [rho * sigma1 * sigma2, sigma2 * sigma2],
            ])
            delta = model - mean
            total += float(delta @ np.linalg.inv(covariance) @ delta)

    return total


def chronometer_chi2(z_grid, e_grid, H0):
    z = HZ_CC[:, 0]
    H_obs = HZ_CC[:, 1]
    H_err = HZ_CC[:, 2]
    H_model = H0 * np.interp(z, z_grid, e_grid)
    return float(np.sum(((H_obs - H_model) / H_err) ** 2))


def information_criteria(chi2, n_params, n_data):
    aic = chi2 + 2 * n_params
    bic = chi2 + n_params * np.log(n_data)
    return aic, bic


def make_dvch_objective(zmax, z_sn, mu_sn, covinv_sn, config):
    def objective(theta):
        H0, omega_m0, n, rd, M_shift = theta
        allowed = (
            50.0 < H0 < 90.0
            and 0.05 < omega_m0 < 0.6
            and 0.0 < n < 1.0
            and 120.0 < rd < 170.0
            and -1.0 < M_shift < 1.0
        )
        if not allowed:
            return 1e50

        try:
            z_grid, e_grid = solve_dvch_grid(
                zmax=zmax,
                H0=H0,
                omega_m0=omega_m0,
                omega_r0=config.omega_r0,
                n=n,
                beta=config.beta,
            )
            return (
                pantheon_chi2(z_grid, e_grid, H0, M_shift, z_sn, mu_sn, covinv_sn)
                + bao_chi2(z_grid, e_grid, H0, rd)
                + chronometer_chi2(z_grid, e_grid, H0)
            )
        except Exception:
            return 1e50

    return objective


def make_lcdm_objective(zmax, z_sn, mu_sn, covinv_sn, config):
    def objective(theta):
        H0, omega_m0, rd, M_shift = theta
        allowed = (
            50.0 < H0 < 90.0
            and 0.05 < omega_m0 < 0.6
            and 120.0 < rd < 170.0
            and -1.0 < M_shift < 1.0
        )
        if not allowed:
            return 1e50

        try:
            z_grid, e_grid = solve_lcdm_grid(
                zmax=zmax,
                omega_m0=omega_m0,
                omega_r0=config.omega_r0,
            )
            return (
                pantheon_chi2(z_grid, e_grid, H0, M_shift, z_sn, mu_sn, covinv_sn)
                + bao_chi2(z_grid, e_grid, H0, rd)
                + chronometer_chi2(z_grid, e_grid, H0)
            )
        except Exception:
            return 1e50

    return objective


def run_joint_fit(config=FitConfig()):
    z_sn, mu_sn, cov_sn = load_pantheon_plus()
    covinv_sn = np.linalg.inv(cov_sn)

    zmax = max(np.max(z_sn), np.max(HZ_CC[:, 0]), max(block["z"] for block in BAO_BLOCKS))
    n_data = len(z_sn) + len(HZ_CC) + sum(1 if block["kind"] == "DV" else 2 for block in BAO_BLOCKS)

    dvch_objective = make_dvch_objective(zmax, z_sn, mu_sn, covinv_sn, config)
    lcdm_objective = make_lcdm_objective(zmax, z_sn, mu_sn, covinv_sn, config)

    res_dvch = minimize(
        dvch_objective,
        x0=np.array(config.dvch_init),
        method="Nelder-Mead",
        options={"maxiter": config.maxiter, "xatol": config.xatol, "fatol": config.fatol},
    )

    res_lcdm = minimize(
        lcdm_objective,
        x0=np.array(config.lcdm_init),
        method="Nelder-Mead",
        options={"maxiter": config.maxiter, "xatol": config.xatol, "fatol": config.fatol},
    )

    chi2_dvch = float(res_dvch.fun)
    chi2_lcdm = float(res_lcdm.fun)

    aic_dvch, bic_dvch = information_criteria(chi2_dvch, n_params=5, n_data=n_data)
    aic_lcdm, bic_lcdm = information_criteria(chi2_lcdm, n_params=4, n_data=n_data)

    print("Joint real-data fit: Pantheon+ + DESI BAO + cosmic chronometers")
    print(f"Fixed assumptions: omega_r0 = {config.omega_r0:.1e}, beta = {config.beta:.1e}")
    print()

    print("DVCH best fit [H0, Om0, n, rd, M_shift]:")
    print(res_dvch.x)
    print(f"chi2 = {chi2_dvch:.6f}")
    print(f"AIC  = {aic_dvch:.6f}")
    print(f"BIC  = {bic_dvch:.6f}")
    print()

    print("LCDM best fit [H0, Om0, rd, M_shift]:")
    print(res_lcdm.x)
    print(f"chi2 = {chi2_lcdm:.6f}")
    print(f"AIC  = {aic_lcdm:.6f}")
    print(f"BIC  = {bic_lcdm:.6f}")
    print()

    print("Relative comparison (DVCH - LCDM):")
    print(f"Delta chi2 = {chi2_dvch - chi2_lcdm:.6f}")
    print(f"Delta AIC  = {aic_dvch - aic_lcdm:.6f}")
    print(f"Delta BIC  = {bic_dvch - bic_lcdm:.6f}")

    return {
        "dvch_result": res_dvch,
        "lcdm_result": res_lcdm,
        "chi2_dvch": chi2_dvch,
        "chi2_lcdm": chi2_lcdm,
        "aic_dvch": aic_dvch,
        "aic_lcdm": aic_lcdm,
        "bic_dvch": bic_dvch,
        "bic_lcdm": bic_lcdm,
        "n_data": n_data,
    }


if __name__ == "__main__":
    run_joint_fit()

Joint real-data fit: Pantheon+ + DESI BAO + cosmic chronometers
Fixed assumptions: omega_r0 = 9.0e-05, beta = 1.0e-04

DVCH best fit [H0, Om0, n, rd, M_shift]:
[ 6.90286397e+01  3.55514687e-01  9.25559443e-02  1.44855945e+02
 -1.27295508e-01]
chi2 = 1481.223284
AIC  = 1491.223284
BIC  = 1518.320187

LCDM best fit [H0, Om0, rd, M_shift]:
[ 6.95625173e+01  3.09755700e-01  1.44721971e+02 -1.20449667e-01]
chi2 = 1484.424786
AIC  = 1492.424786
BIC  = 1514.102309

Relative comparison (DVCH - LCDM):
Delta chi2 = -3.201502
Delta AIC  = -1.201502
Delta BIC  = 4.217879
